In [ ]:
# stratified train-test split or stratified sampling (makes train and test split identical)
from sklearn.model_selection import train_test_split
from src.utils import load_cleaned_csv

cleaned_df = load_cleaned_csv()

train_df, test_df = train_test_split(
    cleaned_df,
    test_size=0.2,
    random_state=42,
    stratify=cleaned_df["label"]
)

print(f"\nTrain:")
print(train_df["label"].value_counts(normalize=True))
print(f"\nTest:")
print(test_df["label"].value_counts(normalize=True))

In [ ]:
# encode labels
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
all_labels = pd.concat([train_df['label'], test_df['label']])
label_encoder.fit(all_labels)

train_df["label_encoded"] = label_encoder.transform(train_df["label"])
test_df["label_encoded"] = label_encoder.transform(test_df["label"])

label_encoder.classes_

In [ ]:
# convert to huggingface dataset
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(test_df)

In [ ]:
# tokenize
from src.utils import tokenize_function

train_tokenized = train_dataset.map(tokenize_function, batched=True)
eval_tokenized = eval_dataset.map(tokenize_function, batched=True)

In [ ]:
# training config
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./legalbert_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3, # change from 3-5 / if overfitting: 1-2
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no",
    fp16=True
)

In [ ]:
# metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",   # for imbalanced data
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch

weights = compute_class_weight(
    class_weight = 'balanced',
    classes=np.unique(train_df["label_encoded"]),
    y=train_df["label_encoded"]
)

# Convert to a Tensor and send to GPU/CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

for cls, w in zip(np.unique(train_df["label_encoded"]), weights):
    label_name = ["civil", "criminal", "legal_fees"][cls]
    print(f"Weight for {label_name} ({cls}): {w}")

In [ ]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        # Get predictions from Legal-BERT
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # 'class_weights'
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# train
from src.loader import model, tokenizer

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
from datetime import datetime
from src.utils import trainer, tokenizer

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")  # Format: 20240124_153045
model_path = f"./models/{timestamp}/legalbert_model_{timestamp}"

# Save
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to: {model_path}")